# Optimize Parn's Experiment with Ax


In [228]:
from ax.api.client import Client
from ax.api.configs import RangeParameterConfig, ChoiceParameterConfig
import pandas as pd

In [229]:
# 1. Initialize the Client.
client = Client(random_seed=42)

In [230]:
# 2. Configure where Ax will search.
client.configure_experiment(
    name="parn_exp",
    parameters=[
        RangeParameterConfig(
            name="L_resin",
            bounds=(0, 50),
            parameter_type="float",
        ),
        RangeParameterConfig(
            name="L_metal",
            bounds=(0, 50),
            parameter_type="float",
        ),
        # ChoiceParameterConfig(
        #     name="M",
        #     values=["Metal", "Resin", "Wood"],
        #     parameter_type="str",
        # ),
    ],
)

In [231]:
client.configure_optimization(objective="-deformation")

# Round 1

In [232]:
suggest_params = []
for i in range(3):
    data = client.get_next_trials(max_trials=1)
    for trial_index, parameters in data.items():
        # print(f"Trial {trial_index} with parameters {parameters}")
        suggest_params.append({"trial_index": trial_index,**parameters})
        # client.complete_trial(
        #     trial_index=trial_index,
        #     raw_data={},
        # )
df_trials = pd.DataFrame(suggest_params)
df_trials["L_wood"] = 100 - df_trials["L_resin"] - df_trials["L_metal"]
df_trials["deformation"] = pd.NA
df_trials["reported"] = False

[INFO 06-16 15:40:29] ax.api.client: GenerationStrategy(name='Center+Sobol+MBM:fast', nodes=[CenterGenerationNode(next_node_name='Sobol'), GenerationNode(name='Sobol', generator_specs=[GeneratorSpec(generator_enum=Sobol, generator_key_override=None)], transition_criteria=[MinTrials(transition_to='MBM'), MinTrials(transition_to='MBM')], suggested_experiment_status=ExperimentStatus.INITIALIZATION, pausing_criteria=[MaxTrialsAwaitingData(threshold=5)]), GenerationNode(name='MBM', generator_specs=[GeneratorSpec(generator_enum=BoTorch, generator_key_override=None)], transition_criteria=None, suggested_experiment_status=ExperimentStatus.OPTIMIZATION, pausing_criteria=None)]) chosen based on user input and problem structure.
[INFO 06-16 15:40:29] ax.api.client: Generated new trial 0 with parameters {'L_resin': 25.0, 'L_metal': 25.0} using GenerationNode CenterOfSearchSpace.
[INFO 06-16 15:40:29] ax.api.client: Generated new trial 1 with parameters {'L_resin': 5.215009, 'L_metal': 34.220961} u

In [233]:
df_trials

,trial_index,L_resin,L_metal,L_wood,deformation,reported
0,0,25.000000,25.000000,50.000000,<NA>,False
1,1,5.215009,34.220961,60.564030,<NA>,False
2,2,27.339656,22.836606,49.823738,<NA>,False


In [234]:
df_trials.loc[0, "deformation"] = 67.63
df_trials.loc[1, "deformation"] = 57.52
df_trials.loc[2, "deformation"] = 64.63

In [235]:
for idx, row in df_trials.iterrows():
    # print(idx, row)
    if not row["reported"]:
        client.complete_trial(
            trial_index=row["trial_index"],
            raw_data={"deformation": float(row["deformation"])},
        )
        df_trials.loc[idx, "reported"] = True
        print(f"Trial {row['trial_index']} completed with deformation {row['deformation']}")

[INFO 06-16 15:40:29] ax.api.client: Trial 0 marked COMPLETED.
[INFO 06-16 15:40:29] ax.api.client: Trial 1 marked COMPLETED.
[INFO 06-16 15:40:29] ax.api.client: Trial 2 marked COMPLETED.


Trial 0 completed with deformation 67.63
Trial 1 completed with deformation 57.52
Trial 2 completed with deformation 64.63


# Round 2

In [236]:
suggest_params = []
for i in range(3):
    data = client.get_next_trials(max_trials=1)
    for trial_index, parameters in data.items():
        # print(f"Trial {trial_index} with parameters {parameters}")
        suggest_params.append({"trial_index": trial_index,**parameters})
        # client.complete_trial(
        #     trial_index=trial_index,
        #     raw_data={},
        # )
_df_trials = pd.DataFrame(suggest_params)
_df_trials["L_wood"] = 100 - _df_trials["L_resin"] - _df_trials["L_metal"]
_df_trials["deformation"] = pd.NA
_df_trials["reported"] = False

df_trials = pd.concat([df_trials, _df_trials], ignore_index=True).reset_index(drop=True)
df_trials

[INFO 06-16 15:40:29] ax.api.client: Generated new trial 3 with parameters {'L_resin': 40.027484, 'L_metal': 41.409247} using GenerationNode Sobol.


[INFO 06-16 15:40:29] ax.api.client: Generated new trial 4 with parameters {'L_resin': 18.305284, 'L_metal': 3.095676} using GenerationNode Sobol.
[INFO 06-16 15:40:30] ax.api.client: Generated new trial 5 with parameters {'L_resin': 0.292806, 'L_metal': 38.618832} using GenerationNode MBM.


,trial_index,L_resin,L_metal,L_wood,deformation,reported
0,0,25.000000,25.000000,50.000000,67.63,True
1,1,5.215009,34.220961,60.564030,57.52,True
2,2,27.339656,22.836606,49.823738,64.63,True
3,3,40.027484,41.409247,18.563269,<NA>,False
4,4,18.305284,3.095676,78.599040,<NA>,False
5,5,0.292806,38.618832,61.088362,<NA>,False


In [ ]:
df_trials.loc[3, "deformation"] = 0
df_trials.loc[4, "deformation"] = 0
df_trials.loc[5, "deformation"] = 0

In [ ]:
for idx, row in df_trials.iterrows():
    # print(idx, row)
    if not row["reported"]:
        client.complete_trial(
            trial_index=row["trial_index"],
            raw_data={"deformation": float(row["deformation"])},
        )
        df_trials.loc[idx, "reported"] = True
        print(f"Trial {row['trial_index']} completed with deformation {row['deformation']}")